# Test inference.py
This notebook tests if inference.py and the VibeShiftInference class work as expected. Follow each section to verify functionality.

## 1. Import Required Libraries
Import torch, numpy, soundfile, and other dependencies required by inference.py. Ensure all packages are installed.

In [ ]:
# Import required libraries
import torch
import numpy as np
import soundfile as sf
import base64
import io
import os
import sys
try:
    import pyloudnorm
except ImportError:
    print('pyloudnorm not installed. Loudness normalization will fallback to peak normalization.')

## 2. Import and Reload inference.py
Import the VibeShiftInference class from inference.py. Reload if needed to reflect code changes.

In [ ]:
# Import VibeShiftInference from inference.py
import importlib
import sys
module_path = os.path.abspath('../inference.py')
if module_path not in sys.path:
    sys.path.append(os.path.dirname(module_path))
spec = importlib.util.spec_from_file_location('inference', module_path)
inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference)
VibeShiftInference = inference.VibeShiftInference

film_conditioner.py STARTED


c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\.venv\lib\site-packages\resampy\filters.py:50: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 3. Initialize VibeShiftInference
Create an instance of VibeShiftInference with sample parameters (checkpoint path, soundfont path, device, etc.).

In [ ]:
# Initialize VibeShiftInference instance
checkpoint_path = 'checkpoints/checkpoints1/best.pt'
soundfont_path = '../data/TimGM6mb.sf2'
device = 'cpu'  # Change to 'cuda' if GPU is available
sample_rate = 44100
model_type = '44khz'
n_quantizers = 9
processor = VibeShiftInference(
    checkpoint_path=checkpoint_path,
    soundfont_path=soundfont_path,
    device=device,
    sample_rate=sample_rate,
    model_type=model_type,
    n_quantizers=n_quantizers
 )

✅ VibeShiftInference initialized on cpu


## 4. Test Audio File Transformation
Call transform_audio() with a sample input WAV file and verify that the output file is created and playable.

In [ ]:
# Debug: Test transform_audio() with a small WAV or FLAC file and memory check
import psutil
import platform
def print_memory():
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / 1024**2
    print(f'Current process memory usage: {mem:.2f} MB')
print(f'Platform: {platform.platform()}')
print_memory()

# Use a small WAV or FLAC file for debugging (update path as needed)
input_audio = r"data/audio_metadata3.wav"  # Try .wav first, change to .flac if needed
output_wav = r"app/outputs/test_output_debug.wav"

import soundfile as sf

# If input is FLAC, convert to WAV in memory for the model if needed
def ensure_wav(input_path):
    if input_path.lower().endswith('.wav'):
        return input_path
    # Convert FLAC to WAV in memory and save as temp file
    import tempfile
    data, sr = sf.read(input_path)
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(tmp.name, data, sr)
    return tmp.name

try:
    input_path = ensure_wav(input_audio)
    print('Before inference:')
    print_memory()
    result_path = processor.transform_audio(
        audio_path=input_path,
        output_path=output_wav,
        target_genre=1,
        num_steps=5,  # Use fewer steps for debugging
        method='heun',
        normalize_method='peak'
    )
    print('After inference:')
    print_memory()
    print(f'Output file created: {result_path}')
    # Optionally, play the output file if in a notebook environment
    import IPython.display as ipd
    display(ipd.Audio(result_path))
except Exception as e:
    import traceback
    print(f'Error during audio file transformation: {e}')
    traceback.print_exc()

Platform: Windows-10-10.0.26100-SP0


NameError: name 'os' is not defined

## 5. Test Audio Array Transformation
Load a WAV file as a numpy array, call process_audio(), and check the output array shape and values.

In [ ]:
# Test process_audio() with numpy array input
try:
    audio_data, sr = sf.read(input_wav)
    if sr != sample_rate:
        import resampy
        audio_data = resampy.resample(audio_data, sr, sample_rate)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    output_array = processor.process_audio(
        audio_input=audio_data,
        target_genre=1,
        num_steps=50,
        method='heun',
        normalize_method='peak'
    )
    print(f'Output array shape: {output_array.shape}')
    # Optionally, play the output audio
    display(ipd.Audio(output_array, rate=sample_rate))
except Exception as e:
    print(f'Error during audio array transformation: {e}')

## 6. Test Frontend Output Format
Call process_for_frontend() with a numpy audio array and verify the returned dictionary contains base64 audio and metadata.

In [ ]:
# Test process_for_frontend() with numpy audio array
try:
    frontend_result = processor.process_for_frontend(
        audio_input=audio_data,
        target_genre=1,
        num_steps=50,
        normalize_method='peak'
    )
    print('Frontend output dictionary:')
    for k, v in frontend_result.items():
        if k == 'audio_data':
            print(f'{k}: [base64 string, length={len(v)}]')
        else:
            print(f'{k}: {v}')
    # Optionally, decode and play the audio
    audio_bytes = base64.b64decode(frontend_result['audio_data'])
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmpfile:
        tmpfile.write(audio_bytes)
        tmpfile.flush()
        display(ipd.Audio(tmpfile.name))
except Exception as e:
    print(f'Error during frontend output test: {e}')

## 7. Cleanup and Memory Management
Delete the VibeShiftInference instance and check that models are properly unloaded from memory.

In [ ]:
# Cleanup: delete processor and check memory
try:
    del processor
    import gc
    gc.collect()
    print('VibeShiftInference instance deleted and memory cleaned up.')
except Exception as e:
    print(f'Error during cleanup: {e}')